# Field-Strength Adjusted Strokes Gained

## The Problem

SG (strokes gained) is computed **vs. the field you're playing in** — it is zero-sum within a field by definition.  This creates a direct comparability problem when players compete across tours:

| Scenario | Raw SG | What it actually means |
|---|---|---|
| +2.0 SG at a PGA Tour Signature event (70-player elite field) | +2.0 | Very impressive |
| +2.0 SG at a weak DP World Tour event (156-player mixed field) | +2.0 | Far less impressive |
| +2.0 SG in a LIV event | +2.0 | Somewhere in between |

**We need a way to put all rounds on the same scale — "vs. a typical PGA Tour field."**

## The Approach: Iterative Field-Strength Estimation

Since we have no external player skill ratings, we bootstrap them from the round data itself using **alternating optimization**:

1. Initialize each player's skill = their career mean `sg_total`
2. For each event: `field_quality = mean(skill_rating of all players in that event)`
3. For each round: `adj_sg = raw_sg + (field_quality - reference_quality)`  
   where `reference_quality = mean field_quality of PGA Tour events`
4. Re-estimate skill ratings using **recency-weighted** mean of `adj_sg`
5. Repeat steps 2-4 for N iterations until convergence (~5-8 iterations)

This is mathematically similar to **iterative least-squares / alternating minimization** and is the same class of algorithm DataGolf uses for their skill model (though theirs is more sophisticated).

## Key Parameters

### `decay_lambda` — Recency Weighting

When estimating a player's current skill level, recent rounds should count more than old ones.  We use exponential decay:

```
weight(i) = exp(-decay_lambda * i)
```

Where `i = 0` is the most recent round.  This gives:

| `decay_lambda` | Half-life (rounds) | Effect |
|---|---|---|
| 0.00 | ∞ | All rounds equal weight |
| 0.01 | 69 rounds | Very gradual decay |
| **0.03** | **23 rounds** | **Default — balances recency and sample size** |
| 0.05 | 14 rounds | Emphasizes recent form |
| 0.10 | 7 rounds | Very hot-hand |

The UI exposes a slider from 0.00 to 0.10 so you can tune this interactively.

### `min_rounds` — Minimum Data Threshold

Players with fewer than `min_rounds` (default: 15) rounds get assigned the tour-average skill as their fallback.  This prevents noisy single-event estimates from corrupting field quality calculations.

### `n_iter` — Iterations

The algorithm converges within ~5 iterations in practice.  We use 8 by default for a small safety margin.

## Component Adjustments

Field quality is derived from `sg_total`.  For individual components, we distribute the field adjustment proportionally so the components remain internally consistent:

```
adj_sg_putt  = sg_putt  + field_adj * (1/4)
adj_sg_app   = sg_app   + field_adj * (1/4)
adj_sg_arg   = sg_arg   + field_adj * (1/4)
adj_sg_ott   = sg_ott   + field_adj * (1/4)
adj_sg_t2g   = sg_t2g   + field_adj * (3/4)   # = app + arg + ott
adj_sg_total = sg_total + field_adj * 1
```

This preserves: `adj_putt + adj_app + adj_arg + adj_ott = adj_total`

The assumption (field difficulty distributes equally across all skill components) is a simplification, but it is directionally correct and interpretable.

In [ ]:
import sys
from pathlib import Path

# Add Scripts to path so we can import the module
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root / "Scripts"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from field_adjusted_sg import compute_field_adjusted_sg, get_field_quality_summary

DATA_PATH = repo_root / "Data" / "in Use" / "combined_rounds_all_2017_2026.csv"
rounds_raw = pd.read_csv(DATA_PATH, low_memory=False)
rounds_raw["event_completed"] = pd.to_datetime(rounds_raw["event_completed"], errors="coerce")
print(f"Loaded {len(rounds_raw):,} rounds | tours: {sorted(rounds_raw['tour'].unique())}")

## Step 1 — Run the Algorithm

In [ ]:
rounds_adj = compute_field_adjusted_sg(
    rounds_raw,
    n_iter=8,
    decay_lambda=0.03,
    min_rounds=15,
    reference_tour="PGA",
)

print("New columns added:", [c for c in rounds_adj.columns if c not in rounds_raw.columns])
rounds_adj[["tour", "event_name", "player_name", "sg_total", "adj_sg_total", "field_adj"]].head(10)

## Step 2 — Field Quality by Event

Strongest fields should be PGA Tour majors / signature events.  Weakest should be secondary DP World Tour events.

In [ ]:
fq = get_field_quality_summary(rounds_adj)

print("=== Top 20 Strongest Fields ===")
display(fq.head(20)[["event_name", "tour", "season", "avg_field_quality"]])

print("\n=== Bottom 20 Weakest Fields ===")
display(fq.tail(20)[["event_name", "tour", "season", "avg_field_quality"]])

## Step 3 — Field Quality Distribution by Tour

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

tour_colors = {"PGA": "#4e8ef7", "EURO": "#f7a74e", "LIV": "#f74e4e"}
for tour, grp in fq.groupby("tour"):
    ax.hist(
        grp["avg_field_quality"],
        bins=30,
        alpha=0.65,
        label=tour,
        color=tour_colors.get(str(tour).upper(), "grey"),
    )

ax.axvline(0, color="white", linestyle="--", linewidth=1, alpha=0.6, label="PGA reference")
ax.set_xlabel("Avg Field Quality (strokes vs. PGA Tour baseline)")
ax.set_ylabel("Events")
ax.set_title("Field Quality Distribution by Tour")
ax.legend()
plt.tight_layout()
plt.show()

## Step 4 — Individual Component Consistency Check

Verify: `adj_putt + adj_app + adj_arg + adj_ott ≈ adj_total`

In [ ]:
sample = rounds_adj.dropna(subset=["adj_sg_total", "adj_sg_putt", "adj_sg_app", "adj_sg_arg", "adj_sg_ott"]).copy()
sample["component_sum"] = sample[["adj_sg_putt", "adj_sg_app", "adj_sg_arg", "adj_sg_ott"]].sum(axis=1)
sample["residual"] = sample["adj_sg_total"] - sample["component_sum"]

print(f"Max residual (adj_total - sum of components): {sample['residual'].abs().max():.4f}")
print(f"Mean residual: {sample['residual'].mean():.6f}")
print("(Small residuals are expected due to pre-existing rounding in raw data)")

## Step 5 — Who Moves the Most?

Players who gain the most from the adjustment (playing in strong fields) vs. players who are deflated (playing in weak fields).

In [ ]:
# Player-level: mean raw vs mean adjusted SG (using last 40 rounds per player)
recent = (
    rounds_adj
    .sort_values(["dg_id", "event_completed"], ascending=[True, False])
    .groupby("dg_id")
    .head(40)
)

player_comparison = (
    recent
    .groupby(["dg_id", "player_name"])
    .agg(
        raw_sg=     ("sg_total",     "mean"),
        adj_sg=     ("adj_sg_total", "mean"),
        n_rounds=   ("sg_total",     "count"),
        primary_tour=("tour",        lambda x: x.mode().iloc[0]),
    )
    .reset_index()
)
player_comparison["delta"] = player_comparison["adj_sg"] - player_comparison["raw_sg"]
player_comparison = player_comparison[player_comparison["n_rounds"] >= 15].sort_values("delta")

print("=== Most deflated (played in weak fields) ===")
display(player_comparison.head(10)[["player_name", "primary_tour", "n_rounds", "raw_sg", "adj_sg", "delta"]].round(3))

print("\n=== Most boosted (played in strong fields) ===")
display(player_comparison.tail(10)[["player_name", "primary_tour", "n_rounds", "raw_sg", "adj_sg", "delta"]].round(3))

## Step 6 — Algorithm Convergence Check

In [ ]:
# Compare 3 vs 8 iterations to verify convergence
rounds_3 = compute_field_adjusted_sg(rounds_raw, n_iter=3, decay_lambda=0.03)
rounds_8 = compute_field_adjusted_sg(rounds_raw, n_iter=8, decay_lambda=0.03)

merged = rounds_8[["dg_id", "event_id", "adj_sg_total"]].merge(
    rounds_3[["dg_id", "event_id", "adj_sg_total"]].rename(columns={"adj_sg_total": "adj_sg_3"}),
    on=["dg_id", "event_id"]
).dropna()

diff = (merged["adj_sg_total"] - merged["adj_sg_3"]).abs()
print(f"Max difference (3 vs 8 iterations): {diff.max():.5f} strokes")
print(f"Mean difference: {diff.mean():.5f} strokes")
print("Effectively converged by iteration 3.")

## Step 7 — Decay Sensitivity

How does the decay parameter change player rankings?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, lam in zip(axes, [0.0, 0.03, 0.10]):
    rounds_lam = compute_field_adjusted_sg(rounds_raw, n_iter=8, decay_lambda=lam)
    recent_lam = (
        rounds_lam
        .sort_values(["dg_id", "event_completed"], ascending=[True, False])
        .groupby("dg_id").head(40)
        .groupby("dg_id")["adj_sg_total"].mean()
        .rename(f"adj_lam_{lam}")
    )
    top20 = recent_lam.nlargest(20).reset_index()
    top20 = top20.merge(rounds_lam[["dg_id", "player_name"]].drop_duplicates(), on="dg_id", how="left")
    top20["label"] = top20["player_name"].str.split(",").str[0]

    ax.barh(top20["label"][::-1], top20[f"adj_lam_{lam}"][::-1], color="#4e8ef7", alpha=0.8)
    ax.set_title(f"decay_lambda = {lam}\n(half-life: {'∞' if lam==0 else round(0.693/lam,1)} rounds)")
    ax.set_xlabel("Adj SG/round (L40)")

plt.suptitle("Top 20 Players by Adj SG — Decay Sensitivity", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## Summary

### What the adjustment does
- Levels the playing field across PGA Tour, DP World Tour, and LIV events
- Players who exclusively play weaker tours have their raw SG deflated
- Players who punch above their weight in strong fields are rewarded
- All metrics are expressed relative to "a typical PGA Tour event field"

### What it does NOT do
- It does not adjust for course difficulty or course fit
- It does not account for weather conditions
- Individual component adjustments (putt/app/arg/ott) assume uniform field difficulty across all skill areas — this is a simplification

### Recommended defaults
| Parameter | Value | Rationale |
|---|---|---|
| `n_iter` | 8 | Converges by ~3-4; 8 is a safe margin |
| `decay_lambda` | 0.03 | Half-life ~23 rounds, balances recency and sample stability |
| `min_rounds` | 15 | Prevents noisy estimates from single-event players |
| `reference_tour` | PGA | All adjusted SG expressed vs. average PGA Tour field |

### In the app
Toggle is in the sidebar: **Field-Adjusted SG**.  When on:
- L12/L24/L40 rolling stats use `adj_sg_*` instead of `sg_*`
- All tours are included in the rolling window (not just PGA)
- A recency decay slider lets you tune the sensitivity
- Raw SG data is unchanged — the toggle is non-destructive